In [7]:
import pandas as pd
import numpy as np
import glob
import os
from minicons import cwe
import torch
from tqdm import tqdm
from sklearn.cluster import KMeans

For the validation analysis we need to relabel the clusters based on new ones we make in relation to all of the lemmas together, hopefully at the lemma level.

In [53]:
# """
# Load the Features
# """


#we have 30 pretty common but not too highly polysemous (<15 senses) words and all of their okens from semcor
# at least 50 tokens each, I think. 

# the data for each word are stored in different files in ./features/semcor. 
# we want to load them all into a single dataframe.

words = ['no', 'first', 'one', 'third', 'large', 'high', 'clear', 'same', 'general', 'ready', 'age', 'information', 'word', 'door', 'meaning', 'government', 'study', 'animal', 'growth', 'building', 'left', 'seem', 'died', 'obtained', 'ran', 'built', 'considered', 'took', 'stand', 'suppose']
corpus = "acl"
# Set your directory path
folder_path = "/home/gsc685/data/features/{}/".format(corpus)
# Get all CSV file paths in the directory
csv_files = [os.path.join(folder_path, word + "_feature_vectors_roberta_buchanan_layer7.csv") for word in words]
print(csv_files)


['/home/gsc685/data/features/acl/no_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/acl/first_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/acl/one_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/acl/third_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/acl/large_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/acl/high_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/acl/clear_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/acl/same_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/acl/general_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/acl/ready_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/acl/age_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/acl/information_feature_vectors_roberta_buchanan_layer

In [54]:

# Read and concatenate all CSV files into a single DataFrame
all_data = pd.concat([pd.read_csv(f).assign(word=w) for f,w in zip(csv_files, words)] )

# Preview
print(all_data.head())

   Unnamed: 0  cluster  feature  predicted_value source  token_id word
0           0        0    FALSE        -0.016429    acl     82099   no
1           1        0     TRUE         0.430595    acl     82099   no
2           2        0        a         0.280720    acl     82099   no
3           3        0  abandon         0.202101    acl     82099   no
4           4        0  abdomen         0.035593    acl     82099   no


In [55]:
len(all_data.token_id.unique())
all_data.token_id = all_data["word"] + all_data["token_id"].astype(str)
len(all_data.token_id.unique())


29088

In [56]:
"""
Load the Sentences and calculate embeddings
"""

# Set your directory path
folder_path = "/home/gsc685/data/collected_tokens/{}/".format(corpus)
# Get all CSV file paths in the directory
csv_files = [os.path.join(folder_path, word + ".csv") for word in words]
# print(csv_files)

all_tokens = pd.concat([pd.read_csv(f).assign(word=w) for f,w in zip(csv_files, words)], ignore_index=True)
all_tokens["token_id"] = all_tokens["word"] + all_tokens["Unnamed: 0"].astype(str)


#it might be that we didnt get data for some sentences so ... lets exclude them. 
indexes_to_keep = all_data['token_id'].unique().tolist()
print(len(all_tokens))
all_tokens = all_tokens[all_tokens["token_id"].isin(indexes_to_keep)]
print(len(all_tokens))



3297700
29088


In [57]:
all_tokens.head()

,Unnamed: 0,corpus_id,sentence_id,sentence,start_idx,end_idx,pos,word,token_id
221,221,2370623,92,"candidates, but we have no way to identify the...",25,27,NaN,no,no221
1167,1167,7063079,97,The experimental group had one month of time t...,128,131,NaN,no,no1167
1191,1191,7715096,21,Evaluation of systems for disambiguating ambig...,100,107,NaN,no,no1191
1259,1259,219300900,49,This seems to make a classification task easie...,61,62,NaN,no,no1259
1302,1302,219301550,32,Another problem is that there is no sense inve...,43,46,NaN,no,no1302


In [58]:
len(all_tokens.sentence_id.unique())
len(all_tokens)


29088

In [59]:
# now you need roberta embeddings for the sentences in each half.
_device = "cuda:1" if torch.cuda.is_available() else "cpu"
_embedding_model = cwe.CWE('roberta-base', device = _device)
_batch_size=75

# helper function to batch process inputs
def batch_iterable(iterable, batch_size):
    for i in range(0, len(iterable), batch_size):
        yield iterable[i:i + batch_size]


def get_embeddings(df):

    # data as list of tuples
    # data = list(zip(sentences, word))

    # save all queries separately
    # (needed because some words do not occur in
    # sentences in the same form and must be fixed first)
    words = df['word']
    sentences = df['sentence']


    queries = list(zip(sentences, words))
    embs = []
    for i, batch in tqdm(enumerate(batch_iterable(queries, _batch_size))):

        batch_embs = _embedding_model.extract_representation(batch, layer=7).cpu().detach().numpy()
        embs.append(batch_embs)

    return np.vstack(embs)

embs = get_embeddings(all_tokens)

388it [01:24,  4.62it/s]


In [60]:
"""
clusterize - but according to the whole dataset. 
"""

# first -- how many unique lemmas are there?
print(len(all_tokens["word"].unique()))

# there are 30 so we'll need 30 clusters
k_means_n = 30

def cluster(embeddings):
    """
    input: list of roberta embeddings of a single layer
    output: list of cluster IDs. 
    """
    kmeans_obj = KMeans(n_clusters=k_means_n, n_init=10)
    kmeans_obj.fit(embeddings)

    #label_list = kmeans_obj.labels_
    #cluster_centroids = kmeans_obj.cluster_centers_

    clusters = kmeans_obj.fit_predict(embeddings)
    return clusters

clusters = cluster(embs)

30


In [61]:
# add the cluster info to the tokens dataframes
all_tokens['cluster'] = clusters

In [62]:
# save these lists with clusters to disk

all_tokens.to_csv('/home/gsc685/data/validation_tokens_acl.csv')


# not doing anything below this anymore